# 📈 Time-Series Sales Forecasting System

**What this notebook does:**
1. Installs all required libraries
2. Mounts Google Drive (to save trained models)
3. Uploads your dataset
4. Trains four models per US state: SARIMA · Prophet · XGBoost · LSTM
5. Evaluates and compares models (RMSE, MAE, MAPE)
6. Selects the best model per state automatically
7. Generates 8-week forecasts
8. Saves everything to Google Drive

**Before running:** Go to `Runtime → Change runtime type → T4 GPU`


## ✅ Step 1 — Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print('✅ GPU is available!')
    print(result.stdout[:500])
else:
    print('⚠️  No GPU detected. Go to Runtime → Change runtime type → T4 GPU')
    print('LSTM will still train but slower on CPU.')

## 📦 Step 2 — Install Libraries

In [ ]:
# This cell installs everything needed. Takes ~3 minutes on first run.
!pip install -q prophet xgboost statsmodels scikit-learn openpyxl
!pip install -q tensorflow==2.15.0
print('✅ All libraries installed!')

## 💾 Step 3 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR   = '/content/drive/MyDrive/forecasting_project'
ARTIFACTS_DIR = f'{PROJECT_DIR}/artifacts'
DATA_DIR      = f'{PROJECT_DIR}/data'

os.makedirs(ARTIFACTS_DIR, exist_ok=True)
os.makedirs(DATA_DIR,      exist_ok=True)
print(f'✅ Google Drive mounted. Project folder: {PROJECT_DIR}')

## 📂 Step 4 — Upload Dataset

In [ ]:
from google.colab import files
import shutil

print('Please upload your Excel file (sales_data.xlsx / Forecasting_Case-_Study.xlsx)')
uploaded = files.upload()

for fname in uploaded.keys():
    dest = f'{DATA_DIR}/sales_data.xlsx'
    shutil.copy(fname, dest)
    print(f'✅ Uploaded and saved to: {dest}')

## 🔧 Step 5 — Preprocessing & Feature Engineering

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

# ── Load data ────────────────────────────────────────────────────────────────
df = pd.read_excel(f'{DATA_DIR}/sales_data.xlsx')
df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
df['date'] = pd.to_datetime(df['date'])
if 'total' in df.columns:
    df.rename(columns={'total': 'sales'}, inplace=True)

print(f'Dataset shape: {df.shape}')
print(f'States:        {df["state"].nunique()}')
print(f'Date range:    {df["date"].min().date()} → {df["date"].max().date()}')
print(f'Columns:       {df.columns.tolist()}')
df.head()

In [ ]:
# ── Preprocessing functions ──────────────────────────────────────────────────

def clean_and_resample(df, state, freq='W'):
    """Filter to one state and resample to a regular weekly series."""
    s = df[df['state'] == state].copy()
    s = s.sort_values('date').drop_duplicates('date')
    s = s.set_index('date')[['sales']]
    weekly = s.resample(freq).sum()
    weekly['sales'] = weekly['sales'].replace(0, np.nan)
    weekly['sales'] = weekly['sales'].interpolate(method='linear').ffill().bfill()
    weekly.reset_index(inplace=True)
    weekly.rename(columns={'date': 'ds', 'sales': 'y'}, inplace=True)
    return weekly


FEATURE_COLS = [
    'lag_1', 'lag_4', 'lag_8', 'lag_13',
    'rolling_mean_4', 'rolling_mean_13', 'rolling_std_4',
    'week_of_year', 'month', 'quarter', 'year', 'trend',
]


def add_features(df):
    """Add lag and calendar features for ML models."""
    df = df.copy().sort_values('ds').reset_index(drop=True)
    y = df['y']
    df['lag_1']  = y.shift(1)
    df['lag_4']  = y.shift(4)
    df['lag_8']  = y.shift(8)
    df['lag_13'] = y.shift(13)
    df['rolling_mean_4']  = y.shift(1).rolling(4).mean()
    df['rolling_mean_13'] = y.shift(1).rolling(13).mean()
    df['rolling_std_4']   = y.shift(1).rolling(4).std()
    df['week_of_year'] = df['ds'].dt.isocalendar().week.astype(int)
    df['month']        = df['ds'].dt.month
    df['quarter']      = df['ds'].dt.quarter
    df['year']         = df['ds'].dt.year
    df['trend']        = np.arange(len(df))
    df.dropna(inplace=True)
    df.reset_index(drop=True, inplace=True)
    return df


def train_test_split_ts(df, test_weeks=8):
    split = len(df) - test_weeks
    return df.iloc[:split].copy(), df.iloc[split:].copy()


def make_future_dates(last_date, weeks=8):
    future_dates = pd.date_range(start=last_date + pd.Timedelta(weeks=1),
                                 periods=weeks, freq='W')
    return pd.DataFrame({'ds': future_dates})


def compute_metrics(actual, predicted):
    from sklearn.metrics import mean_absolute_error, mean_squared_error
    actual    = np.array(actual, dtype=float)
    predicted = np.array(predicted, dtype=float)
    rmse = float(np.sqrt(mean_squared_error(actual, predicted)))
    mae  = float(mean_absolute_error(actual, predicted))
    mask = actual != 0
    mape = float(np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100)
    return {'rmse': round(rmse, 2), 'mae': round(mae, 2), 'mape': round(mape, 4)}


# Quick sanity check on one state
sample = clean_and_resample(df, 'California')
print(f'California weekly series: {len(sample)} observations')
print(sample.tail(3))

## 🤖 Step 6 — Model Training Functions

We define one function per model. Each returns `(metrics_dict, future_forecast_list)`.

In [ ]:
# ── SARIMA ────────────────────────────────────────────────────────────────────

def train_sarima(train_df, test_df, forecast_weeks=8):
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    model = SARIMAX(
        train_df['y'].values,
        order=(1, 1, 1),
        seasonal_order=(1, 1, 0, 52),
        enforce_stationarity=False,
        enforce_invertibility=False,
    )
    fit = model.fit(disp=False, maxiter=200)
    test_preds   = [max(0, v) for v in fit.forecast(len(test_df))]
    future_preds = [max(0, v) for v in fit.forecast(len(test_df) + forecast_weeks)[-forecast_weeks:]]
    metrics = compute_metrics(test_df['y'].values, test_preds)
    return fit, metrics, future_preds


# ── Prophet ───────────────────────────────────────────────────────────────────

def train_prophet(train_df, test_df, forecast_weeks=8):
    from prophet import Prophet
    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        seasonality_mode='multiplicative',
        changepoint_prior_scale=0.05,
    )
    m.fit(train_df[['ds', 'y']])
    n_total  = len(test_df) + forecast_weeks
    future   = m.make_future_dataframe(periods=n_total, freq='W')
    forecast = m.predict(future)
    test_preds   = [max(0, v) for v in forecast['yhat'].values[-(n_total):-forecast_weeks]]
    future_preds = [max(0, v) for v in forecast['yhat'].values[-forecast_weeks:]]
    metrics = compute_metrics(test_df['y'].values, test_preds)
    return m, metrics, future_preds


# ── XGBoost ───────────────────────────────────────────────────────────────────

def train_xgboost(train_df, test_df, forecast_weeks=8):
    from xgboost import XGBRegressor
    from sklearn.preprocessing import MinMaxScaler

    scaler = MinMaxScaler()
    df_feat = add_features(train_df)
    X_train = df_feat[FEATURE_COLS].values
    y_train = df_feat['y'].values
    X_scaled = scaler.fit_transform(X_train)

    model = XGBRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=5,
        subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1
    )
    model.fit(X_scaled, y_train)

    def recursive_forecast(history_df, n):
        history = history_df.copy()
        preds = []
        for _ in range(n):
            feat = add_features(history)
            if len(feat) == 0:
                preds.append(0.0)
                continue
            row  = feat[FEATURE_COLS].iloc[-1].values.reshape(1, -1)
            row_scaled = scaler.transform(row)
            yhat = max(0.0, float(model.predict(row_scaled)[0]))
            preds.append(yhat)
            new_row = pd.DataFrame({'ds': [history['ds'].iloc[-1] + pd.Timedelta(weeks=1)], 'y': [yhat]})
            history = pd.concat([history, new_row], ignore_index=True)
        return preds

    test_preds   = recursive_forecast(train_df, len(test_df))
    full_history = pd.concat([train_df, test_df], ignore_index=True)
    future_preds = recursive_forecast(full_history, forecast_weeks)
    metrics = compute_metrics(test_df['y'].values, test_preds)
    return (model, scaler), metrics, future_preds


# ── LSTM ──────────────────────────────────────────────────────────────────────

LOOK_BACK = 13

def train_lstm(train_df, test_df, forecast_weeks=8, epochs=60):
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import LSTM, Dense, Dropout
    from tensorflow.keras.callbacks import EarlyStopping
    from sklearn.preprocessing import MinMaxScaler

    scaler = MinMaxScaler(feature_range=(0, 1))
    values = train_df['y'].values.reshape(-1, 1)
    scaled = scaler.fit_transform(values)

    def make_sequences(data, look_back):
        X, y = [], []
        for i in range(look_back, len(data)):
            X.append(data[i-look_back:i, 0])
            y.append(data[i, 0])
        return np.array(X), np.array(y)

    X_tr, y_tr = make_sequences(scaled, LOOK_BACK)
    X_tr = X_tr.reshape(X_tr.shape[0], X_tr.shape[1], 1)

    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(LOOK_BACK, 1)),
        Dropout(0.2),
        LSTM(32),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1),
    ])
    model.compile(optimizer='adam', loss='mse')
    es = EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)
    model.fit(X_tr, y_tr, epochs=epochs, batch_size=16, callbacks=[es], verbose=0)

    def predict_n(seed_seq, n):
        seq   = seed_seq.copy()
        preds = []
        for _ in range(n):
            X    = seq.reshape(1, LOOK_BACK, 1)
            yhat = float(model.predict(X, verbose=0)[0, 0])
            preds.append(yhat)
            seq  = np.append(seq[1:], [[yhat]], axis=0)
        inv = scaler.inverse_transform(np.array(preds).reshape(-1, 1)).flatten()
        return [max(0.0, float(v)) for v in inv]

    last_seq_train  = scaled[-LOOK_BACK:]
    test_seq_start  = scaled[-(LOOK_BACK + len(test_df)):-len(test_df)]
    test_preds      = predict_n(test_seq_start, len(test_df))

    full_vals = np.concatenate([values, test_df['y'].values.reshape(-1,1)])
    full_scl  = scaler.transform(full_vals)
    future_preds = predict_n(full_scl[-LOOK_BACK:], forecast_weeks)

    metrics = compute_metrics(test_df['y'].values, test_preds)
    return (model, scaler), metrics, future_preds


print('✅ Model functions defined.')

## 🏋️ Step 7 — Train All Models for All States

> **Tip:** To test on a single state first, change `STATES_TO_TRAIN` to e.g. `['California', 'Texas']`

In [ ]:
import joblib, json, time, os

# ── Config ────────────────────────────────────────────────────────────────────
FORECAST_WEEKS  = 8
TEST_WEEKS      = 8
TRAIN_LSTM      = True   # Set False to skip LSTM (much faster)

ALL_STATES = sorted(df['state'].unique())
STATES_TO_TRAIN = ALL_STATES  # Change to a small list to test

print(f'Training on {len(STATES_TO_TRAIN)} states …')
print(f'Models: SARIMA + Prophet + XGBoost' + (' + LSTM' if TRAIN_LSTM else ''))

all_results = {}
t_total = time.time()

for state in STATES_TO_TRAIN:
    print(f'\n── {state} ──')
    state_key = state.replace(' ', '_')
    state_dir = f'{ARTIFACTS_DIR}/{state_key}'
    os.makedirs(state_dir, exist_ok=True)

    weekly   = clean_and_resample(df, state)
    train_df, test_df = train_test_split_ts(weekly, TEST_WEEKS)

    state_metrics   = {}
    state_forecasts = {}

    # SARIMA
    try:
        t0 = time.time()
        obj, m, fc = train_sarima(train_df, test_df, FORECAST_WEEKS)
        joblib.dump(obj, f'{state_dir}/sarima.pkl')
        state_metrics['sarima']   = m
        state_forecasts['sarima'] = fc
        print(f'  SARIMA  RMSE={m["rmse"]:>12,.0f}  MAE={m["mae"]:>12,.0f}  MAPE={m["mape"]:>6.2f}%  ({time.time()-t0:.1f}s)')
    except Exception as e:
        print(f'  SARIMA  FAILED: {e}')

    # Prophet
    try:
        t0 = time.time()
        obj, m, fc = train_prophet(train_df, test_df, FORECAST_WEEKS)
        joblib.dump(obj, f'{state_dir}/prophet.pkl')
        state_metrics['prophet']   = m
        state_forecasts['prophet'] = fc
        print(f'  Prophet RMSE={m["rmse"]:>12,.0f}  MAE={m["mae"]:>12,.0f}  MAPE={m["mape"]:>6.2f}%  ({time.time()-t0:.1f}s)')
    except Exception as e:
        print(f'  Prophet FAILED: {e}')

    # XGBoost
    try:
        t0 = time.time()
        obj, m, fc = train_xgboost(train_df, test_df, FORECAST_WEEKS)
        joblib.dump(obj, f'{state_dir}/xgboost.pkl')
        state_metrics['xgboost']   = m
        state_forecasts['xgboost'] = fc
        print(f'  XGBoost RMSE={m["rmse"]:>12,.0f}  MAE={m["mae"]:>12,.0f}  MAPE={m["mape"]:>6.2f}%  ({time.time()-t0:.1f}s)')
    except Exception as e:
        print(f'  XGBoost FAILED: {e}')

    # LSTM
    if TRAIN_LSTM:
        try:
            t0 = time.time()
            obj, m, fc = train_lstm(train_df, test_df, FORECAST_WEEKS)
            model_obj, scaler_obj = obj
            model_obj.save(f'{state_dir}/lstm_model.h5')
            joblib.dump(scaler_obj, f'{state_dir}/lstm_scaler.pkl')
            state_metrics['lstm']   = m
            state_forecasts['lstm'] = fc
            print(f'  LSTM    RMSE={m["rmse"]:>12,.0f}  MAE={m["mae"]:>12,.0f}  MAPE={m["mape"]:>6.2f}%  ({time.time()-t0:.1f}s)')
        except Exception as e:
            print(f'  LSTM    FAILED: {e}')

    # Best model selection
    if not state_metrics:
        print(f'  ⚠️  All models failed for {state}. Skipping.')
        continue

    best_model = min(state_metrics, key=lambda k: state_metrics[k]['rmse'])
    best_fc    = state_forecasts[best_model]
    print(f'  ★ Best: {best_model.upper()} (RMSE={state_metrics[best_model]["rmse"]:,.0f})')

    last_date    = weekly['ds'].max()
    future_dates = make_future_dates(last_date, FORECAST_WEEKS)

    forecast_records = [
        {'date': str(d.date()), 'sales': round(v, 2)}
        for d, v in zip(future_dates['ds'], best_fc)
    ]

    summary = {
        'state':      state,
        'best_model': best_model,
        'metrics':    state_metrics,
        'forecast':   forecast_records,
    }

    with open(f'{state_dir}/summary.json', 'w') as f:
        json.dump(summary, f, indent=2)

    all_results[state] = summary

with open(f'{ARTIFACTS_DIR}/all_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

print(f'\n\n✅ Training complete! Total time: {(time.time()-t_total)/60:.1f} minutes')
print(f'Results saved to: {ARTIFACTS_DIR}/all_results.json')

## 📊 Step 8 — Visualise Results

In [ ]:
# ── Model comparison across all states ───────────────────────────────────────

records = []
for state, data in all_results.items():
    for model, m in data['metrics'].items():
        records.append({'state': state, 'model': model, **m})
metrics_df = pd.DataFrame(records)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric in zip(axes, ['rmse', 'mae', 'mape']):
    pivot = metrics_df.pivot_table(index='state', columns='model', values=metric)
    pivot.mean().sort_values().plot(kind='bar', ax=ax, color=['#4CAF50','#2196F3','#FF9800','#9C27B0'][:len(pivot.columns)])
    ax.set_title(f'Average {metric.upper()} by Model', fontsize=12, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Model Comparison Across All States', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{ARTIFACTS_DIR}/model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Best model wins ───────────────────────────────────────────────────────────
best_counts = {}
for data in all_results.values():
    bm = data['best_model']
    best_counts[bm] = best_counts.get(bm, 0) + 1

plt.figure(figsize=(7, 5))
plt.bar(best_counts.keys(), best_counts.values(),
        color=['#4CAF50','#2196F3','#FF9800','#9C27B0'][:len(best_counts)])
plt.title('Best Model Per State (# of states won)', fontsize=13, fontweight='bold')
plt.ylabel('Number of States')
plt.grid(axis='y', alpha=0.3)
for i, (k, v) in enumerate(best_counts.items()):
    plt.text(i, v + 0.2, str(v), ha='center', fontsize=11)
plt.tight_layout()
plt.savefig(f'{ARTIFACTS_DIR}/best_model_wins.png', dpi=120)
plt.show()

print('Model wins:', best_counts)

In [ ]:
# ── Forecast plot for one state ───────────────────────────────────────────────
EXAMPLE_STATE = 'California'  # Change to any state in your dataset

weekly_ex  = clean_and_resample(df, EXAMPLE_STATE)
result_ex  = all_results[EXAMPLE_STATE]
forecast_records = result_ex['forecast']

fc_df = pd.DataFrame(forecast_records)
fc_df['date'] = pd.to_datetime(fc_df['date'])

plt.figure(figsize=(14, 5))
plt.plot(weekly_ex['ds'], weekly_ex['y'], label='Historical', color='steelblue', linewidth=1.5)
plt.plot(fc_df['date'], fc_df['sales'], label=f'Forecast ({result_ex["best_model"]})',
         color='orange', linewidth=2, linestyle='--', marker='o', markersize=6)
plt.axvline(weekly_ex['ds'].max(), color='red', linestyle=':', alpha=0.6, label='Forecast start')
plt.title(f'{EXAMPLE_STATE} — 8-Week Sales Forecast', fontsize=13, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Sales ($)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{ARTIFACTS_DIR}/{EXAMPLE_STATE.replace(" ","_")}_forecast.png', dpi=120)
plt.show()

## 📋 Step 9 — Summary Table

In [ ]:
rows = []
for state, data in all_results.items():
    bm = data['best_model']
    m  = data['metrics'][bm]
    rows.append({'State': state, 'Best Model': bm.upper(),
                 'RMSE': f"{m['rmse']:,.0f}", 'MAE': f"{m['mae']:,.0f}",
                 'MAPE (%)': f"{m['mape']:.2f}"})

summary_table = pd.DataFrame(rows).sort_values('State')
print(summary_table.to_string(index=False))

## 🗜️ Step 10 — Download Artifacts

Zip and download the entire `artifacts/` folder so you can use it with the local FastAPI backend.

In [ ]:
import shutil
from google.colab import files

zip_path = '/content/artifacts_export'
shutil.make_archive(zip_path, 'zip', ARTIFACTS_DIR)
print(f'✅ Zipped: {zip_path}.zip')
files.download(f'{zip_path}.zip')
print('⬇️  Download started!')